<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/las_%EA%B5%AC%ED%98%84_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [ ]:
class VGCExtractor(nn.Module): #특징 추출기(이미지 기반)

  def __init__(self, input_dim):  # ← 오타 수정 필요 (nn.Module → input_dim)
    super(VGCExtractor, self).__init__()
    self.dim=64
    self.hide_dim=128
    in_channel, freq_dim, out_dim=self.check_dim(input_dim)
    self.in_channel=in_channel
    self.freq_dim=freq_dim
    self.out_dim=out_dim

    self.extractor=nn.Sequential( # b,1,128,80
        nn.Conv2d(in_channel,self.init_dim,3,stride=3,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.init_dim,self.init_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2) #절반으로 줄어듦?


        nn.Conv2d(self.init_dim,self.hide_dim,3,stride=1,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.hide_dim,self.hide_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2) # b,t//4,output_dim
    )

    def check_dim(self,input_dim):
      if input_dim%13==0: #MFCC
        return int(input_dim/13),13,(13//4)*self.hide_dim #TIME, 주파수 단위 크기, 특징 벡터 차원

      elif input_dim%40==0: #LOG MEL
        return int(input_dim/40),40,(40//4)*self.hide_dim

      else:
        raise ValueError('Invalid input dimension')

    def view_input(self,feature,feat_len): #b,t,d -> b,c,t,f
      feat_len=feat_len//4
      if feature.shape[1]%4!=0:
        feature=feature[:,:-(feature.shape[1]%4),:].contiguous()
      bs,ts,ds=feature.shape
      feature=feature.view(bs,ts,self.in_channel,self.freq_dim)
      feature=feature.transpose(1,2)
      return feature, feat_len


    def forward(self,feature, feat_len):
      feature, feat_len=self.view_input(feature,feat_len) #b,t,d -> b,c,t,f
      feature=feature.extractor(feature) #다운샘플링, 고차원 feqture 추출
      feature=feature.transpose(1,2) #b,c,t,f -> b,t,c,f
      feature=feature.contiguous().view(feature.shape[0],feature.shape[1],self.out_dim)
      return feature, feat_len

In [ ]:
class CNNExtractor(nn.Module):
  def __init__(self,input_dim,out_dim):
    super(CNNExtractor,self).__init__()
    self.out_dim=out_dim
    self.extractor=nn.Sequential(
        nn.Conv1d(input_dim,out_dim,4,stride=2,padding=1),
        nn.Conv1d(out_dim,out_dim,4,stride=2,padding=1),
    )

  def forward(self,feature,feat_len):
    feat_len=feat_len//4
    feature=feature.transpose(1,2)
    feature=self.extractor(feature)
    feature=feature.transpose(1,2)
    return feature, feat_len

In [ ]:
class RNNLayer(nn.Module):
  def __init__(self,input_dim,module,dim,bidirection,dropout,layer_norm,sample_rate,sample_style,proj):
    super(RNNLayer,self).__init__()
    rnn_out_dim=2*dim if bidirection else dim
    self.out_dim=sample_rate * rnn_out_dim if sample_rate>1 and sample_style=='concat' else rnn_out_dim
    self.dropout=dropout
    self.layer_norm=layer_norm
    self.sample_rate=sample_rate
    self.sample_style=sample_style
    self.proj=proj

    if self.sample_style not in ['drop','concat']:
      raise ValueError('Unsupported Sample Style: '+self.sample_style)

    self.layer=getattr(nn.module.upper())(
        input_dim,dim,bidirectional=bidirection,num_layer=1,batch_first=True
    )

    if self.layer_norm:
      self.ln=nn.LayerNorm(rnn_out_dim)
    if self.dropout>0:
      self.dp=nn.Dropout(p=dropout)
    if self.proj:
      self.pj=nn.Linear(rnn_out_dim,rnn_out_dim)

  def forward(self,input_x,x_len):
    if not self.training:
      self.layer.flatten_parameters()
    output,_=self.layer(input_x)

    if self.layer_norm:
      output=self.ln(output)

    if self.dropout>0:
      output=self.dp(output)

    if self.sample_rate>1:
      batch_size,timestep,feature_dim=output.shape
      x_len=x_len//self.sample_rate

      if self.sample_style='drop':
        output=output[:,::self.sample_rate,:].contiguous()
      else:
        if timestep%self.sample_rate!=0:
          output=output[:,:-(timestep%self.sample_rate),:]
        output=output.contiguous().view(batch_size,int(timestep/self.sample_rate),feature_dim*self.sample_rate)

    if self.proj:
      output=torch.tanh(self.pj(output))

    return output, x_len

In [ ]:
class BaseAttention(nn.Module):

  def __init__(self,temprature,num_head):
    super().__init__()
    self.temprature=temprature
    self.num_head=num_head
    self.softmax=nn.Softmax(dim=-1)
    self.reset_mem()

  def reset_mem(self):#마스크 초기화
    self.mask=None
    self.k_len=None

  def set_mem(self,prev_att):
    pass

  def compute_mask(self,k,k_len): #마스크 생성 함수
    self.k_len=k_len #b,t,d 패딩을 제외한 길이 확인
    bs,ts,_=k.shape # b,t
    self.mask=np.zeros((bs,self.num_head,ts)) #b, num_head, t
    for idx,sl in enumerate(k_len):# b,t,d
      self.mask[idx, : , sl:]=1 #마스크 범위 1로 변경
    self.maks=torch.from_numpy(self.mask).to(k_len.device,dtype=torch.bool).view(-1,ts)

  def _attend(self,energy, value): #마스크 씌우기
    attn=energy/self.temperature
    attn=attn.masked_fill(self.mask, -np.inf) #마스크 위치에 -inf 지정
    attn=self.softmax(attn)
    output=torch.bmm(attn.unsqueeze(1),value).squeeze(1)
    return output,attn

In [ ]:
class ScaleDotAttention(BaseAttention): # scaledot 연산

  def __init__(self,temperature,num_head):
    super().__init__(temperature,num_head)

  def forward(self,q,k,v):
    ts=k.shape[1]
    energy=torch.bmm(q.unsqueeze(1),k.transpose(1,2)).squeeze(1)#배치 단위 행렬곱
    output,attn=self._attend(energy,v)
    attn=attn.view(-1,self.num_head,ts)

    return output,attn

In [ ]:
class LocationAwareAttention(BaseAttention):
  def __init__(temperature,num_head):
    self.prev_att=None
    self.loc_conv=nn.Conv1d(num_head,kernel_num,kernel_size=2*kernel_size+1,padding=kernel_size, bias=False)
    self.gen_energy=nn.Linear(dim,1)
    self.dim=dim

  def reset_mem(self):
    super().reset_mem()
    self.prev_att=None

  def set_mem(self,prev_att):
    self.prev_att=prev_att

  def forward(self,q,k,v):
    bs_nj,ts,_=k.shape
    bs=bs_nh//self.num_head

    if self.prev_att is None:
      self.prev_att = torch.zeros((bs,self.num_head,ts)).to(k.device)
      for idx,sl in enumerate(self.k_len):
        self.prev_att[idx,:,:sl]=1.0/sl

    loc_content=torch.tanh(self.loc_proj(self.loc_conv(self.prev_att).transpose(1,2)))
    loc_content=loc_content.unsqueeze(1).repeat(1,self.num_head,1,1).view(-1,ts,self.dim)
    q=q.unsqueeze(1)

    energy=self.gen_energy(torch.tanh(k+q+loc_context)).squeeze(2)
    output,attn=self._attend(energy,v)
    attn=attn.view(bs,self.num_head,ts)
    self.prev_att=attn

    return output, attn